<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/evolucao-pos-tcc/notebooks/01-processamento_pln.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

📑 Guia de Execução
⚠️ IMPORTANTE: Sempre que o Runtime (Ambiente de Execução) for reiniciado, as Células 1 e 2 devem ser executadas obrigatoriamente para restabelecer os caminhos do Drive e reinstalar as bibliotecas.

🔄 Fluxo de Dependências:
Sessão Recém-Iniciada: Executar Célula 1 ➔ Célula 2.

Primeira vez no projeto: Executar Célula 1 ➔ Célula 2 ➔ Célula 3 (Carga).

Retomando Processamento: Se o banco já existe no Drive, pule a Célula 3 e vá direto para a Célula 4 e/ou 5 e/ou 6.

⚠️ Para a correta execução da Célula 1, é necessário observar a estrutura de subpastas no Google Drive, dentro de uma pasta principal. Atualmente, a estrutura principal é **..\pln\evolucao-pos-tcc** e deve ser armazenada na raíz do Google Drive (MyDrive)):
*   deve haver uma subpasta chamada **\data**, onde será criado automaticamente o banco SQLite (um arquivo com a extensão .db, denominado **base-dados.db**);
*   deve haver uma subpasta chamada **\sql**, onde deverão ser adicionados os dois arquivos .sql responsáveis pela criação das tabelas e população dos dados no banco SQLite. Os arquivos podem ser obtidos junto ao projeto no Github, na pasta **\sql**, sendo  **01-schema.sql** e **02-seed_data.sql**.



In [ ]:
# Célula 1: Montagem do Google Drive e Configuração de Caminhos
from google.colab import drive
import os

# 1. Montagem Segura: Só executa se ainda não estiver montado
if not os.path.exists('/content/drive/MyDrive'):
    print("📂 Montando Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive já está montado e acessível.")

# 2. Configuração Estrita de Caminhos
DRIVE_DIR = '/content/drive/MyDrive/pln/evolucao-pos-tcc'
DB_FILE_NAME = 'data/base-dados.db'
DB_PATH = os.path.join(DRIVE_DIR, DB_FILE_NAME)

# Artefatos SQL
SCHEMA_SQL = os.path.join(DRIVE_DIR, 'sql/01-schema.sql')
SEED_SQL = os.path.join(DRIVE_DIR, 'sql/02-seed_data.sql')

# Pasta de Saída (Outputs)
EXPORT_PATH = os.path.join(DRIVE_DIR, 'outputs')
if not os.path.exists(EXPORT_PATH):
    os.makedirs(EXPORT_PATH)
    print(f"📁 Pasta de exportação criada em: {EXPORT_PATH}")

print(f"📍 Banco de Dados: {DB_PATH}")

⚠️ Na Célula 2 ocorre a criação das estruturas (tabelas, colunas, etc) no banco de dados SQLite.

In [ ]:
# Célula 2: Instalação das bibliotecas e inicialização da estrutura (Schema)

# 1. Instalação Silenciosa
!pip install -q transformers torch pandas bertopic pysentimiento spacy
!python -m spacy download pt_core_news_lg -q

import sqlite3
import torch

# 2. Hardware Check para BERTimbau/BERTopic
device = 0 if torch.cuda.is_available() else -1

def inicializar_estrutura_db(db_path, schema_path):
    """Garante que a estrutura de tabelas esteja presente."""
    print(f"🛠️ Verificando integridade das tabelas...")

    # Se o arquivo de banco não existir, o SQLite o criará automaticamente
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        with open(schema_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())
        conn.commit()
        print("✅ Estrutura (Schema) validada com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao processar Schema: {e}")
    finally:
        conn.close()

# 3. Execução
inicializar_estrutura_db(DB_PATH, SCHEMA_SQL)

print(f"\n🚀 Ambiente pronto (GPU: {'Ativa' if device == 0 else 'Inativa'}).")

⚠️ Na Célula 3, os dados são populados no banco de dados SQLite.

In [ ]:
# Célula 3: Carga Inicial de Dados (Seed SQL)
def executar_carga_dados(db_path, seed_path):
    """Popula o banco apenas se a tabela 'verso' estiver vazia."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        # Verifica se já existem dados para evitar duplicidade no Drive
        cursor.execute("SELECT count(*) FROM verso")
        total_existente = cursor.fetchone()[0]

        if total_existente > 0:
            print(f"ℹ️ O banco já contém {total_existente} versos. Carga inicial ignorada.")
            return

        print("🌱 Semeando dados iniciais (02-seed_data.sql)... Isso pode levar alguns minutos.")
        with open(seed_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())

        conn.commit()
        print(f"✅ Carga de {seed_path} concluída com sucesso!")

    except sqlite3.OperationalError as e:
        print(f"⚠️ Erro operacional: {e}. Verifique se a Célula 2 foi executada.")
    except Exception as e:
        print(f"❌ Erro crítico na carga: {e}")
    finally:
        conn.close()

# Executa a carga (Somente se necessário)
executar_carga_dados(DB_PATH, SEED_SQL)

⚠️ Nas Células 4, 5 e 6 ocorre o processamento do corpus bíblico, verso por verso, e a armazenagem dos dados nas tabelas que registram os detalhes do processamento. Nesta versão do pipeline, a cada execução, as tabelas são limpas e novos dados são inseridos. Não há, portanto, um histórico de execuções que permita comparar os resultados caso haja mudanças nas regras e/ou lógica do processamento.

In [ ]:
# Célula 4: Limpeza Estrutural e Filtro de Densidade (Antidoto ao Ruído Nominal)
import spacy
import sqlite3
import pandas as pd
import re

# Carrega o modelo de português
try:
    nlp = spacy.load("pt_core_news_lg")
except:
    !python -m spacy download pt_core_news_lg
    nlp = spacy.load("pt_core_news_lg")

def limpar_texto_estrutural(texto):
    if not texto or len(texto.strip()) < 3: return "RUIDO_CURTO"

    doc = nlp(texto)

    # Filtramos tokens válidos (substantivos, verbos, adjetivos e nomes próprios)
    # Ignoramos stop words e pontuação
    tokens = [t for t in doc if not t.is_stop and not t.is_punct and t.pos_ in ['NOUN', 'VERB', 'ADJ', 'PROPN']]

    if not tokens: return "RUIDO_VAZIO"

    # Métrica 1: Densidade de Nomes Próprios (PROPN)
    # Se mais de 70% do conteúdo significativo forem nomes próprios, é provavelmente uma lista/genealogia
    propn_count = len([t for t in tokens if t.pos_ == 'PROPN'])
    propn_ratio = propn_count / len(tokens)

    # Métrica 2: Presença de Ação/Estado
    # Antídotos existenciais raramente são frases sem verbos ou adjetivos
    has_action_or_state = any(t.pos_ in ['VERB', 'ADJ'] for t in tokens)

    # CASO CRÍTICO (Ex: Nesias e Hatifa):
    # Verso curto, alta proporção de nomes próprios e sem verbo
    if len(tokens) <= 3 and propn_ratio > 0.5 and not has_action_or_state:
        return "RUIDO_NOMINAL"

    # Retornamos o texto limpo (usando o texto original para preservar a semântica)
    return " ".join([t.text.lower() for t in tokens])

# Execução e Persistência
conn = sqlite3.connect(DB_PATH)
df_versos = pd.read_sql_query("SELECT id, texto FROM verso", conn)

print("🧼 Limpando textos e aplicando filtros de densidade gramatical...")
df_versos['texto_limpo'] = df_versos['texto'].apply(limpar_texto_estrutural)

cursor = conn.cursor()
cursor.execute("DELETE FROM verso_limpo")
df_versos[['id', 'texto_limpo']].rename(columns={'id': 'verso_id'}).to_sql(
    'verso_limpo',
    conn,
    if_exists='append', # Importante: 'append' preserva a estrutura e índices
    index=False
)

conn.commit()
conn.close()
print("✅ Célula 4 concluída! Ruídos nominais e estruturais pré-identificados.")

In [ ]:
# Célula 5: Classificação por Eixos Existenciais, com Persistência de Métricas
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from transformers import pipeline
import sqlite3
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# 1. Configuração e Carga de Dados
conn = sqlite3.connect(DB_PATH)
query = """
    SELECT v.id as verso_id, v.texto, l.abreviacao, g.id as genero_id, vl.texto_limpo
    FROM verso v
    JOIN verso_limpo vl ON v.id = vl.verso_id
    JOIN livro l ON l.id = v.livro_id
    JOIN genero_literario g ON g.id = l.genero_id
"""
df_input = pd.read_sql_query(query, conn)

# Tratamento de erro de tipo
df_input['texto'] = df_input['texto'].fillna('vazio').astype(str)
docs_para_classificar = [str(doc).strip() if (doc and str(doc).strip() != "") else "vazio" for doc in df_input['texto'].tolist()]

# 2. Inicialização do Modelo BERTimbau (Zero-Shot)
embedding_model = pipeline("feature-extraction", model="neuralmind/bert-base-portuguese-cased", device=0)

descricoes_eixos = [
    "Estados de Fadiga Existencial e Alívio da Alma: Descreve o peso da existência, o estar psicologicamente sobrecarregado, o desânimo e o abatimento espiritual. Inclui sentimentos de angústia profunda, o clamor por socorro e descanso e a incapacidade física ou mental. Em contrapartida, oferece o fortalecimento do exausto, o vigor para o que não tem forças, a renovação das energias, o suporte prático nas aflições, o alívio que restaura a vitalidade, o refrigério da paz interior, o descanso para a alma atribulada, o consolo nas aflições e o alívio emocional que torna o fardo da vida leve e suave.",
    "Fragilidade Humana e Firmeza Espiritual: Aborda a brevidade da vida, a impermanência dos dias e a insegurança das coisas mundanas que perecem como a erva. Em oposição, destaca a Rocha Eterna, o fundamento espiritual inabalável, a confiança inabalável em Deus e a segurança de quem constrói a identidade sobre valores eternos, imutáveis e constantes, independentemente das circunstâncias externas.",
    "Crise de Sentido e Vocação Existencial: O sentimento de que tudo é vaidade e a futilidade de uma vida sem significado. Trata do vácuo existencial e da desorientação mental. Contrastando com isso, apresenta o ser chamado por Deus, a eleição divina, o propósito da vocação e missão de vida, a descoberta de um sentido para a vida, o chamado vocacional, os planos de esperança futura, o valor intrínseco do indivíduo e a compreensão de uma missão ou razão de ser que transcende a existência biológica ou material.",
    "Registros Narrativos, Leis e Informações Factuais: Conteúdo puramente informativo, administrativo ou instrutivo. Inclui genealogias, listas de nomes, medidas técnicas, rituais, censos e relatos de viagens. Abrange fundamentalmente fórmulas de introdução de diálogos e marcadores de transição narrativa como 'disse', 'respondeu', 'falou', 'perguntou', servindo como uma categoria técnica para textos sem carga emocional ou existencial direta."
]

model_topic = BERTopic(
    embedding_model=embedding_model,
    zeroshot_topic_list=descricoes_eixos,
    zeroshot_min_similarity=0.1,
    calculate_probabilities=True,
    vectorizer_model=CountVectorizer(ngram_range=(1, 2))
)

print("🤖 Classificando via Zero-Shot e extraindo matriz de probabilidades...")
topics, probs_matrix = model_topic.fit_transform(docs_para_classificar)

# 3. Processamento de Decisões e Preparação do DataFrame Final
rows_to_persist = []

print("⚖️ Processando métricas e aplicando thresholds dinâmicos...")

for i, row in tqdm(df_input.iterrows(), total=len(df_input), desc="Processando Versículos", unit="v"):
    p_ex = probs_matrix[i][0]
    p_tr = probs_matrix[i][1]
    p_va = probs_matrix[i][2]
    p_na = probs_matrix[i][3]

    # Criar uma lista com todas as probabilidades para análise de incerteza
    todas_probs = sorted([p_ex, p_tr, p_va, p_na], reverse=True)

    # 1. Gap de Confiança (Diferença entre o 1º e o 2º lugar)
    gap = todas_probs[0] - todas_probs[1]

    # 2. Entropia Simplificada (Métrica de dispersão)
    # Quanto mais próximo de 0, mais concentrado em um eixo.
    # Quanto mais alto, mais "espalhado" entre os eixos.
    entropia = -sum([p * np.log(p + 1e-9) for p in [p_ex, p_tr, p_va, p_na]])

    existenciais = [p_ex, p_tr, p_va]
    best_idx = np.argmax(existenciais)
    best_score = existenciais[best_idx]

    margem = best_score - p_na
    gen_id = row['genero_id']
    t_len = len(row['texto'])

    decisao_id = 3 # Default: Narrativo
    status = "Descarte"

    # --- Lógica Híbrida de Decisão ---

    # Regra 1: Filtro de Confiança para versos curtos
    if t_len < 35 and margem < 0.25:
        decisao_id = 3
        status = "Filtro Brevidade"

    # Regra 2: Thresholds por Gênero
    else:
        if gen_id in [1, 2]: # Pentateuco/Histórico
            if best_score > 0.88 and margem > 0.15:
                decisao_id, status = best_idx, "Rigor Máximo"

        elif gen_id in [3, 4]: # Poético/Profético
            if best_score > 0.50:
                decisao_id, status = best_idx, "Sensibilidade Poética"

        elif gen_id in [5, 6]: # Evangelhos/Epístolas
            if best_score > 0.60 or (best_score > 0.45 and margem > 0.10):
                decisao_id, status = best_idx, "Resgate/Consolo"

        else: # Geral (7)
            if best_score > 0.75:
                decisao_id, status = best_idx, "Padrão Geral"

    # Preparar objeto para o DataFrame
    rows_to_persist.append({
        'verso_id': row['verso_id'],
        'topico_id': int(decisao_id),
        'p_exaustao': float(p_ex),
        'p_transitoriedade': float(p_tr),
        'p_vazio': float(p_va),
        'p_narrativo': float(p_na),
        'similaridade_final': float(best_score if decisao_id != 3 else p_na),
        'margem_dominancia': float(margem),
        'status_decisao': status,
        'entropia': float(entropia),
        'gap_confianca': float(gap)
    })

# 4. Persistência Final no SQLite
print("💾 Persistindo classificação e métricas no banco de dados...")
df_final = pd.DataFrame(rows_to_persist)

cursor = conn.cursor()
cursor.execute("DELETE FROM verso_topico")
cursor.execute("DELETE FROM topico")

# Mapeamento de eixos para a tabela 'topico'
mapa_eixos = {0: "Exaustão vs. Refrigério", 1: "Transitoriedade vs. Solidez", 2: "Vazio vs. Propósito", 3: "Narrativo/Normativo"}
df_mapa = pd.DataFrame([{'id': k, 'antidoto_referencia': v} for k, v in mapa_eixos.items()])

df_mapa.to_sql('topico', conn, if_exists='append', index=False)
df_final.to_sql('verso_topico', conn, if_exists='append', index=False)

conn.commit()
conn.close()
print(f"✨ Processamento concluído! {len(df_final)} versículos processados.")

⚠️ A Célula 6, que processa a análise de sentimento, é a mais demorada do pipeline. Pode levar mais de 1h até o término.

In [ ]:
# Célula 6: Análise de Sentimento Contextual e Cruzamento Existencial
from pysentimiento import create_analyzer
import pandas as pd
import sqlite3
from tqdm.auto import tqdm

# 1. Inicializar o Analisador
print("🚀 Carregando modelo Transformer para Sentimento (PT-BR)...")
# O analisador 'sentiment' para 'pt' é baseado em BERTimbau, ideal para o TCC
analyzer = create_analyzer(task="sentiment", lang="pt")

# 2. Busca do texto original e dos tópicos
conn = sqlite3.connect(DB_PATH)
df_input = pd.read_sql_query("""
    SELECT v.id as verso_id, v.texto, vt.topico_id
    FROM verso v
    JOIN verso_topico vt ON v.id = vt.verso_id
""", conn)

textos = df_input['texto'].tolist()
verso_ids = df_input['verso_id'].tolist()

# 3. Execução da análise em lotes (Aproveitando a GPU se disponível)
print(f"📊 Analisando carga emocional de {len(textos)} versículos...")
sentimentos = []
batch_size = 64
mapa_num = {'POS': 1, 'NEU': 0, 'NEG': -1}

# O predict em lote é significativamente mais rápido no Colab
for i in tqdm(range(0, len(textos), batch_size)):
    lote = textos[i:i + batch_size]
    ids_lote = verso_ids[i:i + batch_size]
    preds_lote = analyzer.predict(lote)

    for idx, p in enumerate(preds_lote):
        # Capturamos as probabilidades brutas para análises de incerteza se necessário
        sentimentos.append({
            'verso_id': ids_lote[idx],
            'label': p.output,
            'sentimento_num': mapa_num.get(p.output, 0),
            'score_pos': p.probas.get('POS', 0),
            'score_neg': p.probas.get('NEG', 0),
            'score_neu': p.probas.get('NEU', 0)
        })

df_sent = pd.DataFrame(sentimentos)

# 4. Persistência dos Resultados
try:
    cursor = conn.cursor()
    # Limpamos para garantir que a nova classificação da Célula 5 seja a única presente
    cursor.execute("DELETE FROM verso_sentimento")

    # Inserimos os novos resultados (Integridade referencial com 'verso_id')
    df_sent.to_sql('verso_sentimento', conn, if_exists='append', index=False)
    conn.commit()
    print("\n✅ Célula 6 concluída! Sentimentos processados e salvos com sucesso.")

    # 5. RESULTADO FINAL: O DIAGNÓSTICO (PROBLEMA) VS. A CURA (ANTÍDOTO)
    print("\n📈 RESUMO EXECUTIVO: PROBLEMÁTICA (CRISE) VS. ANTÍDOTO (CURA)")

    res_final = pd.read_sql_query("""
        SELECT
            t.antidoto_referencia as Eixo_Filosofico,
            COUNT(*) as Total_Versos,
            SUM(CASE WHEN vs.sentimento_num = 1 THEN 1 ELSE 0 END) as Antidotos_Cura,
            SUM(CASE WHEN vs.sentimento_num = -1 THEN 1 ELSE 0 END) as Problematica_Crise,
            ROUND(AVG(vs.sentimento_num), 3) as Polaridade_Media
        FROM verso_topico vt
        JOIN topico t ON vt.topico_id = t.id
        JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
        WHERE t.id != 3 -- Foco nos eixos Han, Bauman e Frankl
        GROUP BY t.antidoto_referencia
        ORDER BY Polaridade_Media DESC
    """, conn)

    # Exibe a tabela formatada no Colab
    display(res_final)

except Exception as e:
    print(f"❌ Erro na persistência: {e}")
finally:
    conn.close()

⚠️ Ao final do processamento, uma das formas para consultar os dados persistidos no banco SQLite, no arquivo **base-dados.db**, é baixar o arquivo do Google Drive, armazenad na pasta **\data** para a máquina local e utilizar uma ferramenta/console que possibilite a execução de consultas SQL SELECT aos dados nas tabelas que armazenam o processamento. Ou, construir scripts Phyton que acessem o banco de dados e gerem tabelas ou gráficos para a análise dos dados.